# PRACTICA FINAL DE VISIÓN ARTIFICIAL

En la carpeta data existen una serie imágenes de vías de tráfico con etiquetas de objetos relevantes para conducción autónoma: 
- human
- trafficcone
- bicycle
- truck
- car
- bus
- motorcycle


## Carga de datos y conversión

Convertimos las imágenes de la carpeta **data** de formato COCO a YOLO

In [1]:
import os
import json
from collections import defaultdict


In [2]:

# =========================
# CONFIGURACIÓN
# =========================

COCO_JSON = "labels.json"
IMAGES_DIR = "../data/train/data"
LABELS_DIR = "../data/train"
YOLO_LABELS_DIR = os.path.join(LABELS_DIR, "yolo_labels")
IMAGES_SPLIT_DIR = os.path.join(LABELS_DIR, "images_split")


In [3]:


# =========================
# CARGAR COCO
# =========================
# montar ruta completa al archivo JSON
coco_json_path = os.path.join(LABELS_DIR, COCO_JSON)
# cargar el archivo JSON
with open(coco_json_path, "r") as f:
    coco = json.load(f)


In [4]:
images = coco["images"]
annotations = coco["annotations"]
categories = coco["categories"]

In [ ]:
# =========================================
# MAPEO DE CLASES
# =========================================

# YOLO necesita IDs consecutivos desde 0
category_mapping = {}

# Excluimos la clase raíz "traffic-dataset"
valid_categories = [c for c in categories if c["name"] != "traffic-dataset"]

cat_id_to_yolo = {
    c["id"]: idx
    for idx, c in enumerate(valid_categories)
}

for new_id, category in enumerate(valid_categories):
    category_mapping[category["id"]] = new_id

print("Mapeo de clases:")
for category in valid_categories:
    print(f'{category["name"]} -> {category_mapping[category["id"]]}')

# =========================================
# AGRUPAR ANOTACIONES POR IMAGEN
# =========================================

annotations_by_image = defaultdict(list)

for ann in annotations:
    annotations_by_image[ann["image_id"]].append(ann)

Mapeo de clases:
bicycle -> 0
bus -> 1
car -> 2
human -> 3
motorcycle -> 4
trafficcone -> 5
truck -> 6


### Conversión de COCO a YOLO

In [6]:
# =========================================
# CONVERSIÓN
# =========================================

for image in images:

    image_id = image["id"]
    filename = image["file_name"]
    width = image["width"]
    height = image["height"]

    label_filename = os.path.splitext(filename)[0] + ".txt"
    label_path = os.path.join(YOLO_LABELS_DIR, label_filename)

    yolo_lines = []

    for ann in annotations_by_image[image_id]:

        category_id = ann["category_id"]

        # Ignorar categorías no válidas
        if category_id not in category_mapping:
            continue

        class_id = category_mapping[category_id]

        # COCO bbox = [x_min, y_min, width, height]
        x_min, y_min, box_width, box_height = ann["bbox"]

        # Convertir a centro
        x_center = x_min + box_width / 2
        y_center = y_min + box_height / 2

        # Normalizar
        x_center /= width
        y_center /= height
        box_width /= width
        box_height /= height

        line = (
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}"
        )

        yolo_lines.append(line)

    # Guardar .txt
    with open(label_path, "w") as f:
        f.write("\n".join(yolo_lines))

print("\nConversión completada.")


Conversión completada.


### Estratificación

In [7]:
pip install scikit-multilearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import numpy as np
from skmultilearn.model_selection import iterative_train_test_split

TRAIN_RATIO = 0.8

num_classes = len(valid_categories)

# =========================================
# CLASES PRESENTES POR IMAGEN
# =========================================

image_classes = defaultdict(set)

for ann in annotations:
    image_classes[ann["image_id"]].add(
        category_mapping[ann["category_id"]]
    )

# =========================================
# MATRIZ MULTILABEL
# =========================================

X = []
Y = []

for img in images:

    image_id = img["id"]

    labels = np.zeros(num_classes)

    for cls in image_classes[image_id]:
        labels[cls] = 1

    X.append([image_id])
    Y.append(labels)

X = np.array(X)
Y = np.array(Y)

# =========================================
# SPLIT ESTRATIFICADO
# =========================================

X_train, Y_train, X_val, Y_val = iterative_train_test_split(
    X,
    Y,
    test_size=(1 - TRAIN_RATIO)
)

train_ids = set(X_train.flatten())
val_ids = set(X_val.flatten())

print("Train:", len(train_ids))
print("Val:", len(val_ids))

Train: 42280
Val: 10571


In [11]:
import shutil
# =====================================================
# CREAR DIRECTORIOS
# =====================================================

for split in ["train", "val"]:

    os.makedirs(
        os.path.join(IMAGES_SPLIT_DIR, "images", split),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(IMAGES_SPLIT_DIR, "labels", split),
        exist_ok=True
    )

# =====================================================
# CONVERTIR Y GUARDAR LABELS YOLO
# =====================================================

for img in images:

    image_id = img["id"]

    filename = img["file_name"]

    width = img["width"]
    height = img["height"]

    # Determinar split
    if image_id in train_ids:
        split = "train"
    else:
        split = "val"

    # =================================================
    # COPIAR IMAGEN
    # =================================================

    src_image = os.path.join(IMAGES_DIR, filename)

    dst_image = os.path.join(
        IMAGES_SPLIT_DIR,
        "images",
        split,
        filename
    )

    shutil.copy(src_image, dst_image)

    # =================================================
    # CREAR LABEL YOLO
    # =================================================

    label_filename = os.path.splitext(filename)[0] + ".txt"

    label_path = os.path.join(
        IMAGES_SPLIT_DIR,
        "labels",
        split,
        label_filename
    )

    yolo_lines = []

    for ann in annotations_by_image[image_id]:

        cat_id = ann["category_id"]

        if cat_id not in cat_id_to_yolo:
            continue

        class_id = cat_id_to_yolo[cat_id]

        # COCO bbox
        x_min, y_min, box_w, box_h = ann["bbox"]

        # YOLO center format
        x_center = x_min + box_w / 2
        y_center = y_min + box_h / 2

        # Normalizar
        x_center /= width
        y_center /= height
        box_w /= width
        box_h /= height

        line = (
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_w:.6f} "
            f"{box_h:.6f}"
        )

        yolo_lines.append(line)

    # Guardar txt
    with open(label_path, "w") as f:
        f.write("\n".join(yolo_lines))

print("\nDataset YOLO generado correctamente.")

# =====================================================
# GENERAR data.yaml
# =====================================================

class_names = [
    c["name"]
    for c in valid_categories
]

yaml_path = os.path.join(IMAGES_SPLIT_DIR, "data.yaml")

with open(yaml_path, "w") as f:

    f.write(f"path: {IMAGES_SPLIT_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n\n")

    f.write(f"nc: {len(class_names)}\n")

    f.write("names:\n")

    for idx, name in enumerate(class_names):
        f.write(f"  {idx}: {name}\n")

print("data.yaml generado.")


Dataset YOLO generado correctamente.
data.yaml generado.
